In [158]:
"""
functions derived from the exploration of tag rulings .ipyn file
functions to match and filter the section of the ruling where a decision is made
functions to classify the ruling in accordance to the decisions made
functions to classify specific context of rulings in accordance to the decisions made for further processing

"""

#import section
import json
import re
import glob
import os
from collections import defaultdict
import unicodedata

In [159]:
#Normalization functions to strip accents, capitalize characters. 
#Relevant for data preparation
def strip_accents(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c)
    )


def norm_tipo(s: str) -> str:
    return re.sub(r"\s+", " ", strip_accents(s or "")).strip().upper()


In [181]:
#Functions to open json files that include data of the judicial process including the text of the ruling
def open_rulings_json(file_path: str):
    """
        Function that opens a json file in accordance to the provided file_path
        Returns: file_name (judicial process number) and the data loaded from the json file.
    """
    #print(f"file path being processed with processed: {file_path}/n:")
    with open(file_path, 'rb') as file:
        if file.read(2) != b'[]':
            file_name = os.path.basename(file_path).strip('.json')
            #print(file_name)
            with open(file_path, "r", encoding="utf-8") as f:
                try:
                    json_data = json.load(f)
                    return file_name, json_data
                except json.JSONDecodeError:
                    print(f"Error: {file_name} is not a valid JSON file.")
        else:
            print("file came empty so the case did not reach a ruling\n")
            #to avoid a crash of other function due to empty json files
            return "empty", "empty"

def retrieve_ruling(json_data) -> str:
    """
        Function that retrieves the ruling from the json data of a judial case, using the "actividad" tag
        Returns the text of the ruling of the relevant judicial process
    """
    for item in json_data:
        ruling = item.get("actividad")
    return ruling
    

In [182]:
def slicing_rulings(ruling: str, number_of_chars: int):
    """
        Function that matches the section of the ruling in which the decision is made
        Returns the sliced section of the ruling where a decision is made to classify the type of ruling
        Returns the sliced section of the ruling (context of ruling) before the decision is made within a specific number of chars
    """
    #Base Regex to identify the boilerplate phrase prior to which the decision is made
    RE_DECISION_BLOCK = re.compile(
        r"DECISION\b.{0,400}?ADMINISTRANDO\s+JUSTICIA[\s,]+(?:EN\s+)?NOMBRE\s+DEL\s+PUEBLO\s+SOBERANO\s+"
        r"DEL\s+ECUADOR[\s,]*(?:Y\s+)?POR\s+AUTORIDAD\s+DE\s+LA\s+CONSTITUCION\s+Y\s+"
        r"LAS\s+LEYES\s+DE\s+LA\s+REPUBLICA[\s,]*(.*)", 
        re.DOTALL
    )
    #alternative Regex for cases in which no DECISION is found before the boilerplate phrase

    RE_ADMINISTRANDO_BLOCK = re.compile(
        r"ADMINISTRANDO\s+JUSTICIA[\s,]+(?:EN\s+)?"
        r"NOMBRE\s+DEL\s+PUEBLO\s+SOBERANO\s+DEL\s+ECUADOR[\s,]*(?:Y\s+)?"
        r"POR\s+AUTORIDAD\s+DE\s+LA\s+CONSTITUCION\s+Y\s+"
        r"LAS\s+LEYES\s+DE\s+LA\s+REPUBLICA[\s,]*(.*)", 
        re.DOTALL
    )

    #Formating the ruling to match the section in which the decision is made
    clean_decision_ruling = norm_tipo(ruling)

    #Determine whether we have a match and therefore identified the bolierplate phrase where the decision is made
    match = RE_DECISION_BLOCK.search(clean_decision_ruling)

    if match:
        #print("we have a match\n")
        #retrieve the part where the type of decision can be matched
        decision_text = match.group(1)
        formated_decision_text = norm_tipo(decision_text)
        
        #retrieve the context before the decision is made for processing: 

        #index at which the decision section starts with boilerplate phrases
        decision_start = match.start()
        #end of the context ruling just before the decision section starts
        end_ruling_context = decision_start - (len(formated_decision_text) - len(ruling))
        #begining of the ruling context determined in accordance to a desired length of chars
        start_ruling_context = max(0, decision_start - number_of_chars)
        #Slicing the ruling context for processing:
        context_before_decision = ruling[start_ruling_context:end_ruling_context]

        #Return the formated decision text for further matching and the context before the decision
        #print(formated_decision_text[:10], context_before_decision[:10])
        return formated_decision_text, context_before_decision

    else:
        #Try to match in cases were the boilerplate phrase is not preceded by a DECISION section
        alt_match = RE_ADMINISTRANDO_BLOCK.search(clean_decision_ruling)

        if alt_match:
            decision_text = alt_match.group(1)
            formated_decision_text = norm_tipo(decision_text)
            
            #retrieve the context before the decision is made for processing: 
    
            #index at which the decision section starts with boilerplate phrases
            decision_start = alt_match.start()
            #end of the context ruling just before the decision section starts
            end_ruling_context = decision_start - (len(formated_decision_text) - len(ruling))
            #begining of the ruling context determined in accordance to a desired length of chars
            start_ruling_context = max(0, decision_start - number_of_chars)
            #Slicing the ruling context for processing:
            context_before_decision = ruling[start_ruling_context:end_ruling_context]
    
            #Return the formated decision text for further matching and the context before the decision
            #print(formated_decision_text[:10], context_before_decision[:10])
            return formated_decision_text, context_before_decision

        else:
            #No match could be found using both regex alternatives
            print("we couldnt find the boilerplate phrase\n")
            #if no match happened, to avoid failures of the model we still need to return strings 
            #this will be classified as a no match in the dictionary for treacability and correction purposes
            fail_to_match = "failed to find and match the boilerplate decision text"
            failed_context= "no context before decision, no classification possible"
            return fail_to_match, failed_context

def matching_type_of_ruling_and_classify(general_dict: dict, ruling_classifier_dict: dict, file_name: str, formated_decision_text: str, context_before_decision:str):
    """
        Function that matches the decision section with a type of ruling (accept, denied, partial) or a default no match
        In accordance to the matching it classifies each ruling and also counts the number of each ruling type found
    """ 
    #Regex to match the three types of rulings from the taxpayer's perspective:
    #Accept the judicial challenge, deny the judicial challenge or partially accept the judicial challenge. 
    RE_PARTIAL = re.compile(
        r"ACEPT\w*\s+PARCIAL|PARCIALMENTE|ACCOGER?\s+PARCIAL"
    )
    
    RE_ACCEPT = re.compile(
        r"\b(ACEPT\w*|ACOG\w*)\s+(LA\s+|SU\s+)?DEMANDA"
        r"|\bSE\s+ACEPTA\b"
        r"|\bDECLARAR\s+(LA\s+)?VALIDEZ\b"  
    )
    
    RE_DENY = re.compile(
        r"\b(NIEG\w*|NEG\w*|RECHAZ\w*|DESECH\w*)\s+(LA\s+|SU\s+)?DEMANDA"
        r"|\bSIN\s+LUGAR\b"
        r"|\bIMPROCEDENTE\b"
        r"|\bDECLARAR\s+SIN\s+LUGAR\b"
    )

    #matching section
    if RE_PARTIAL.search(formated_decision_text):
        general_dict["partial"] += 1
        #print(f"partial {d}")
        #once confirmed the ruling type:
        ruling_classifier_dict["partial"][file_name] = context_before_decision

    elif RE_ACCEPT.search(formated_decision_text):
        general_dict["accept"] += 1
        #print(f"accept {d}")
        #once confirmed extract a section of the tax ruling:
        ruling_classifier_dict["accept"][file_name] = context_before_decision

    elif RE_DENY.search(formated_decision_text):
        general_dict["denied"] += 1
        #print(f"denied {d}")
        #once confirmed extract a section of the tax ruling:
        ruling_classifier_dict["denied"][file_name] = context_before_decision
    else:
        general_dict["no matches found"] += 1
        ruling_classifier_dict["no matches found"][file_name] = context_before_decision


    #return the dictionaries:
    return general_dict, ruling_classifier_dict

   

    

In [183]:
#Processing and taggin and classifying routine

#path to filtered tax ruling json files (path leverages on volume mounting)
path_to_files = "/home/jovyan/data/processed/"

# retrieval of json files
json_files = glob.glob(os.path.join(path_to_files, "*.json"))

In [184]:
# creating dictionaries for counting ruling types and classifying rulings accordingly
general_dict = {"accept": 0, "partial": 0, "denied": 0, "no matches found": 0}
ruling_classifier_dict = defaultdict(dict)
#specify the desired number of chars for the ruling_context
number_of_chars = 12000
#empty ruling counter:
empty_ruling=0

In [185]:
#logic implementation: 
for file_path in json_files:
    file_name, json_data = open_rulings_json(file_path)
    if file_name != "empty":
        ruling = retrieve_ruling(json_data)
        formated_decision_text, context_before_decision= slicing_rulings(ruling, number_of_chars)
        general_dict, ruling_classifier_dict =  matching_type_of_ruling_and_classify(general_dict, ruling_classifier_dict, file_name, formated_decision_text, context_before_decision)
    else:
        empty_ruling+=1
    

In [186]:
print(general_dict)
distinct_counts = {key: len(set(values)) for key, values in ruling_classifier_dict.items()}
print(f"likely no ruling issued{empty_ruling}")

{'accept': 0, 'partial': 0, 'denied': 0, 'no matches found': 0}
likely no ruling issued0


In [187]:
assert general_dict["accept"] == distinct_counts["accept"]

KeyError: 'accept'

In [188]:
assert general_dict["partial"] == distinct_counts["partial"]

KeyError: 'partial'

In [156]:
assert general_dict["denied"] == distinct_counts["denied"]

In [157]:
assert general_dict["no matches found"] == distinct_counts["no matches found"]